# RoB2 RAG Chatbot

Same RAG pipeline as `grade_rag.ipynb` — only the source URLs and system prompt differ.

**Source:** [Risk of Bias 2.0 Tool](https://www.riskofbias.info/welcome/rob-2-0-tool)

**RoB2 domains covered:**
1. Bias arising from the randomization process
2. Bias due to deviations from intended interventions
3. Bias due to missing outcome data
4. Bias in measurement of the outcome
5. Bias in selection of the reported result

In [2]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from dotenv import load_dotenv
load_dotenv()

from rag_core import load_or_build_index, rag_answer, build_llm

## 1. Configuration

In [ ]:
ROB2_URLS = [
    "https://www.riskofbias.info/welcome/rob-2-0-tool",
    "https://www.riskofbias.info/welcome/rob-2-0-tool/current-version-of-rob-2",
    "https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-for-cluster-randomized-trials",
    "https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-for-crossover-trials",
]

ROB2_SYSTEM_PROMPT = (
    "You are an expert assistant specialising in the Cochrane Risk of Bias tool 2.0 (RoB2). "
    "RoB2 assesses risk of bias in randomized trials across five domains: "
    "(1) randomization process, (2) deviations from intended interventions, "
    "(3) missing outcome data, (4) outcome measurement, (5) selection of the reported result. "
    "Answer the user's question based ONLY on the retrieved RoB2 content below. "
    "Do not fabricate or speculate beyond what is in the context. "
    "If the context does not contain enough information, say so explicitly. "
    "Use precise RoB2 terminology (signalling questions, bias judgments: Low/Some concerns/High, etc.)."
)

CACHE_DIR = ".faiss_cache/rob2"

## 2. Build / Load Index

First run scrapes all RoB2 pages and saves a local FAISS cache.  
Subsequent runs load from cache instantly.

> **Note:** some subpages on riskofbias.info may not be JavaScript-rendered.  
> If a page returns empty text, increase `wait_time` below.

In [13]:
import shutil
shutil.rmtree(".faiss_cache/rob2", ignore_errors=True)                                               
print("Cache cleared.")

Cache cleared.


In [14]:
index = load_or_build_index(ROB2_URLS, cache_path=CACHE_DIR, wait_time=6)
llm = build_llm()
print("Ready.")

Building index from scratch...
  Scraping: https://www.riskofbias.info/welcome/rob-2-0-tool
    -> 6 chunks
  Scraping: https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-in-more-detail
    -> 1 chunks
  Scraping: https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domains-and-questions
    -> 1 chunks
  Scraping: https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domain-1-bias-arising-from-the-randomization-process
    -> 1 chunks
  Scraping: https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domain-2-bias-due-to-deviations-from-intended-interventions
    -> 1 chunks
  Scraping: https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domain-3-bias-due-to-missing-outcome-data
    -> 1 chunks
  Scraping: https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domain-4-bias-in-measurement-of-the-outcome
    -> 1 chunks
  Scraping: https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-domain-5-bias-in-selection-of-the-reported-result
    -> 1 chunks
  Scraping: https://www.r

## 3. Inspect Scraped Content

Verify the index contains meaningful RoB2 text before evaluation.

In [5]:
# Quick sanity check: retrieve top chunks for a known RoB2 concept
probe_query = "signalling questions randomization domain"
docs = index.similarity_search(probe_query, k=3)
for i, doc in enumerate(docs, 1):
    print(f"--- Chunk {i} (source: {doc.metadata['source']}) ---")
    print(doc.page_content[:400])
    print()

--- Chunk 1 (source: https://www.riskofbias.info/welcome/rob-2-0-tool) ---
he RoB 2.0 tool for cluster-randomized trials
The tool (cluster-randomized trials) itself
Blank template for completing the tool, which is currently available in one version
Implement RoB 2.0 for cluster-randomized trials when interest is in the effect of assignment to intervention
Archived: Crossover trials (individually randomized)
Note that a
revised version for crossover trials
is now availabl

--- Chunk 2 (source: https://www.riskofbias.info/welcome/rob-2-0-tool) ---
Risk of bias tools - RoB 2 tool
Search this site
Embedded Files
Skip to main content
Skip to navigation
RoB 2 tool
A revised Cochrane risk of bias tool for randomized trials
A revised tool to assess risk of bias in randomized trials (RoB 2)
Welcome to the website for the RoB 2 tool.
The
current version
(22 August 2019),
suitable for individually-randomized, parallel-group trials.
A version for
clu

--- Chunk 3 (source: https://www.riskofbias.in

## 4. Ask a Question

In [6]:
query = "What are the five domains assessed in RoB2?"
answer, contexts = rag_answer(query, index, llm, k=5, system_prompt=ROB2_SYSTEM_PROMPT)
print("Answer:\n", answer)
print("\n--- Retrieved contexts ---")
for i, ctx in enumerate(contexts, 1):
    print(f"[{i}] {ctx[:200]}...")

Answer:
 The provided context does not contain information about the five domains assessed in RoB2.

--- Retrieved contexts ---
[1] -over trials) (contains macros)
View videos:
RoB 2.0 tool Part 1
,
RoB 2.0 tool Part 2
,
RoB 2.0 tool Part 3
,
RoB 2.0 tool Part 4
.
Google Sites
Report abuse
This site uses cookies from Google to del...
[2] RoB 2.0 for cross-over trials when interest is in the effect of assignment to intervention
Implement RoB 2.0 for cross-over trials when the interest is in the effect of starting and adhering to interv...
[3] he RoB 2.0 tool for cluster-randomized trials
The tool (cluster-randomized trials) itself
Blank template for completing the tool, which is currently available in one version
Implement RoB 2.0 for clus...
[4] Risk of bias tools
Search this site
Embedded Files
Skip to main content
Skip to navigation
404
The page you have entered does not exist
Go to site home
Google Sites
Report abuse
This site uses cookies...
[5] Risk of bias tools
Search this site


## 5. Sanity-Check Questions

In [7]:
sanity_questions = [
    "In my risk of bias assessment, can I add an extra domain corresponding to the quality of statistical results presentation?",
    "Do the different risk of bias domains have different implications in the overall result classification?",
    "I have five domains classified as 'Low risk of bias' and one domain classified as 'High risk of bias'. What is the overall risk of bias judgement?",
    "Is it adequate to stop risk of bias assessment once one of the domains is judged at high risk of bias?",
    "In my systematic review, I am assessing an educational intervention. Can I assume upfront that it is not possible to implement allocation concealment?",
    # "What makes an outcome measurement blinded vs unblinded in RoB2?",
    # "What is selective outcome reporting and which domain covers it?",
    # "When would you rate a trial as 'High' risk of bias for Domain 2?",
    # "What is the overall RoB2 judgment if one domain is rated High?",
    # "How does RoB2 differ from the original Cochrane Risk of Bias tool (RoB1)?",
]

for q in sanity_questions:
    ans, _ = rag_answer(q, index, llm, k=5, system_prompt=ROB2_SYSTEM_PROMPT)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}")
    print()

Q: In my risk of bias assessment, can I add an extra domain corresponding to the quality of statistical results presentation?
A: The provided context does not contain enough information to answer whether an extra domain corresponding to the quality of statistical results presentation can be added to the RoB 2 tool. The context only describes the existing five domains of the RoB 2 tool and its different versions for various tr

Q: Do the different risk of bias domains have different implications in the overall result classification?
A: The provided context describes the RoB2 tool, its citation, and available versions for different trial designs (individually-randomized, parallel-group, cluster-randomized, and crossover trials). It also lists the five domains of the RoB2 tool: (1) randomization process, (2) deviations from intended

Q: I have five domains classified as 'Low risk of bias' and one domain classified as 'High risk of bias'. What is the overall risk of bias judgement?
A: The 

In [9]:
docs = index.similarity_search("risk of bias domain signalling questions", k=5)                      
for i, doc in enumerate(docs, 1):                                                                    
      print(f"--- Chunk {i} ---")                                                                      
      print(f"Source: {doc.metadata['source']}")                                                       
      print(doc.page_content[:300])                                                                    
      print()  

--- Chunk 1 ---
Source: https://www.riskofbias.info/welcome/rob-2-0-tool
Risk of bias tools - RoB 2 tool
Search this site
Embedded Files
Skip to main content
Skip to navigation
RoB 2 tool
A revised Cochrane risk of bias tool for randomized trials
A revised tool to assess risk of bias in randomized trials (RoB 2)
Welcome to the website for the RoB 2 tool.
The
current vers

--- Chunk 2 ---
Source: https://www.riskofbias.info/welcome/rob-2-0-tool
iting the tool
The revised tool may be cited as:
Sterne JAC, Savović J, Page MJ, Elbers RG, Blencowe NS, Boutron I, Cates CJ, Cheng H-Y, Corbett MS, Eldridge SM, Hernán MA, Hopewell S, Hróbjartsson A, Junqueira DR, Jüni P, Kirkham JJ, Lasserson T, Li T, McAleenan A, Reeves BC, Shepperd S, Shrier I, 

--- Chunk 3 ---
Source: https://www.riskofbias.info/welcome/rob-2-0-tool/rob-2-in-more-detail
Risk of bias tools
Search this site
Embedded Files
Skip to main content
Skip to navigation
404
The page you have entered does not exist
Go to site home
Goog

In [11]:
from rag_core import fetch_visible_text  
text = fetch_visible_text("https://www.riskofbias.info/welcome/rob-2-0-tool", wait_time=10)          
print(text[:3000])

Risk of bias tools - RoB 2 tool
Search this site
Embedded Files
Skip to main content
Skip to navigation
RoB 2 tool
A revised Cochrane risk of bias tool for randomized trials
A revised tool to assess risk of bias in randomized trials (RoB 2)
Welcome to the website for the RoB 2 tool.
The
current version
(22 August 2019),
suitable for individually-randomized, parallel-group trials.
A version for
cluster-randomized trials
is now available (10 November 2020, revised 18 March 2021).
A version for
crossover trials
is now available (8 December 2020, revised 18 March 2021).
We are also maintaining an archive of the previous version, which had variants for three different trial designs (see below).
Citing the tool
The revised tool may be cited as:
Sterne JAC, Savović J, Page MJ, Elbers RG, Blencowe NS, Boutron I, Cates CJ, Cheng H-Y, Corbett MS, Eldridge SM, Hernán MA, Hopewell S, Hróbjartsson A, Junqueira DR, Jüni P, Kirkham JJ, Lasserson T, Li T, McAleenan A, Reeves BC, Shepperd S, Shrier I, 

In [12]:
import time
from selenium import webdriver                                                                       
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager                                             
from bs4 import BeautifulSoup                             
                                                                                                    
options = webdriver.ChromeOptions()
options.add_argument("--headless")                                                                   
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get("https://www.riskofbias.info/welcome/rob-2-0-tool")                                       
time.sleep(10)                                                                                       
soup = BeautifulSoup(driver.page_source, "html.parser")                                              
driver.quit()                                                                                        
                                                        
# Print all internal links                                                                           
links = set()                                             
for a in soup.find_all("a", href=True):
    href = a["href"]                                                                                 
    if "riskofbias" in href or href.startswith("/welcome"):
        links.add(href)                                                                              
                                                        
for link in sorted(links):                                                                           
    print(link)   

/welcome
/welcome/home
/welcome/home/original-2016-version-of-robins-i
/welcome/home/original-2016-version-of-robins-i/robins-i-detailed-guidance-2016
/welcome/home/original-2016-version-of-robins-i/robins-i-template-2016
/welcome/home/original-2016-version-of-robins-i/robins-i-tool-2016
/welcome/home/read-more
/welcome/home/the-team
/welcome/rob-2-0-tool
/welcome/rob-2-0-tool/archive-rob-2-0-2016
/welcome/rob-2-0-tool/archive-rob-2-0-cluster-randomized-trials-2016
/welcome/rob-2-0-tool/archive-rob-2-0-cross-over-trials-2016
/welcome/rob-2-0-tool/current-version-of-rob-2
/welcome/rob-2-0-tool/rob-2-for-cluster-randomized-trials
/welcome/rob-2-0-tool/rob-2-for-crossover-trials
/welcome/rob-me-tool
/welcome/robins-e-tool
/welcome/robins-i-v2
/welcome/robvis-visualization-tool


## 6. Convenience Functions (for evaluation notebooks)

In [ ]:
def ask_rob2(question: str) -> str:
    answer, _ = rag_answer(question, index, llm, system_prompt=ROB2_SYSTEM_PROMPT)
    return answer

def get_rob2_contexts(question: str, k: int = 5) -> list[str]:
    _, contexts = rag_answer(question, index, llm, k=k, system_prompt=ROB2_SYSTEM_PROMPT)
    return contexts

## 7. Optional Gradio UI

In [ ]:
import gradio as gr

def gradio_fn(question, history):
    answer, _ = rag_answer(question, index, llm, system_prompt=ROB2_SYSTEM_PROMPT)
    return answer

demo = gr.ChatInterface(
    fn=gradio_fn,
    title="RoB2 Assistant",
    description="Ask questions about the Cochrane Risk of Bias 2.0 tool. Answers are grounded in official RoB2 documentation.",
)
# Uncomment to launch:
# demo.launch()